# Live-Coupled SWAN + GA Array Optimization

---

## Overview

This notebook implements a **fully coupled** SWAN + Genetic Algorithm workflow for optimizing WEC array layouts using **physically resolved wave fields**. Unlike the standalone GA notebook (which assumes a uniform wave field), here the SWAN model provides a spatially varying 2-D wave field that accounts for bathymetric refraction, depth attenuation, shoaling, and wind forcing.

The pipeline is **region-agnostic**: set the environment variable ``SWAN_REGION`` to ``california_bay`` or ``oman`` (or change ``REGION`` in the first code cell below). All paths and time windows are pulled from ``code/regions.py`` so no other edits are required.

The fitness function includes an **explicit on-land / out-of-bounds penalty** so the optimizer cannot place WECs on cells that SWAN flagged as dry.


In [1]:
import json
import os
import sys
import numpy as np
from scipy.special import j0
from scipy.interpolate import RegularGridInterpolator

%matplotlib inline

In [ ]:
# ---------------------------------------------------------------------
# REGION SELECTION — pick the SWAN model to drive this notebook
# ---------------------------------------------------------------------
_search = os.path.abspath(os.getcwd())
for _ in range(10):
    if os.path.isdir(os.path.join(_search, 'code')):
        REPO_ROOT = _search
        break
    _search = os.path.dirname(_search)
else:
    raise FileNotFoundError('Could not locate repository root (code/ directory).')

CODE_DIR = os.path.join(REPO_ROOT, 'code')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from regions import get_region, list_regions, summarize
import swan_runner

REGION = os.environ.get('SWAN_REGION', 'california_bay')
cfg = get_region(REGION)
print(summarize(REGION))

# Paths
MODEL_DIR      = cfg.model_dir
INPUT_TEMPLATE = cfg.swan_input_path
BEST_SITE_JSON = os.path.join(
    cfg.figs_dir('03_site_selection'),
    f'best_site_{cfg.key}.json',
)
BLOCK_FILE     = f'ga_block_{cfg.key}.mat'

SWAN_CMD   = 'swanrun'
SWAN_NPROC = 6

if not os.path.exists(INPUT_TEMPLATE):
    print(f"WARNING: SWAN input template not found: {INPUT_TEMPLATE}")

# Per-region figures directory for this notebook
FIGS_DIR = cfg.figs_dir('04_optimization')
print(f'Figures will be saved to: {FIGS_DIR}')


## 1. Load Best Site from AHP-TOPSIS

The best deployment site was identified by the AHP-TOPSIS workflow and saved to `best_site.json`. This provides the geographic center and baseline wave conditions for the deployment area.

In [ ]:
# Region-specific default if the AHP-TOPSIS output file is missing.
_default_sites = {
    'california_bay': dict(lat=37.60, lon=-122.40, mean_hs=1.5,
                           mean_tp=10.0, mean_power=20.0),
    'oman':           dict(lat=23.50, lon=58.80, mean_hs=1.0,
                           mean_tp=7.0,  mean_power=8.0),
}

if os.path.exists(BEST_SITE_JSON):
    with open(BEST_SITE_JSON) as f:
        site = json.load(f)
    print(f"Loaded best site from {os.path.basename(BEST_SITE_JSON)}")
else:
    site = _default_sites.get(cfg.key, _default_sites['california_bay'])
    print(f"best_site JSON not found; using default for {cfg.key}.")

print(f"Site: ({site['lat']:.3f}N, {site['lon']:.3f}E)")
print(f"Hs={site['mean_hs']:.2f} m  Tp={site['mean_tp']:.2f} s  "
      f"P={site['mean_power']:.1f} kW/m")


## 2. q-Factor and Array Power Functions

Same interaction model as the standalone GA notebook, but here `P_field` is a **callable** that returns the spatially varying isolated WEC power at any `(x, y)` position within the deployment domain, interpolated from the SWAN output grid.

In [4]:
def q_factor(d, k, alpha=0.8, decay=0.05):
    """Hydrodynamic interaction factor between two WECs."""
    if d < 1e-6:
        return 1.0
    kd = k * d
    return max(1.0 + alpha * j0(kd) * np.exp(-decay * kd), 0.1)


def array_power(pos, k, P_field, k_field=None):
    """
    Array power using spatially-varying isolated power from SWAN field.

    pos     : (N, 2) WEC positions in metres within local domain.
    k       : dominant wavenumber (rad/m).
    P_field : callable(x, y) -> isolated WEC power (kW) at (x, y).
    """
    n = len(pos)
    powers = np.zeros(n)
    for i in range(n):
        P_iso_i = P_field(pos[i])
        if n == 1:
            powers[i] = P_iso_i
        else:
            qs = sum(q_factor(np.linalg.norm(pos[i] - pos[j]), k)
                     for j in range(n) if j != i)
            powers[i] = P_iso_i * qs / (n - 1)
    tot = powers.sum()
    P_iso_sum = sum(P_field(pos[i]) for i in range(n))
    q_arr = tot / P_iso_sum if P_iso_sum > 0 else 1.0
    return tot, q_arr


def spacing_pen(pos, d_min):
    """Quadratic penalty for WEC pairs closer than d_min."""
    n = len(pos)
    p = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            d = np.linalg.norm(pos[i] - pos[j])
            if d < d_min:
                p += (d_min - d) ** 2
    return p

## 3. Genetic Algorithm

Same GA structure as the standalone notebook (truncation selection, uniform crossover, uniform mutation), but the fitness function now queries the SWAN-derived wave power interpolator.

In [ ]:
def run_ga(n_wecs, P_field, k, domain, d_min,
           pop_size=120, n_gen=300, mut_rate=0.15,
           valid_mask_fn=None,
           bounds_penalty=1.0e5,
           land_penalty=1.0e5,
           seed=None):
    """
    Genetic-algorithm search for a WEC array layout.

    The fitness function is

        fit = total_power
              − 1e4 · spacing_penalty(d_min)
              − bounds_penalty · (# WECs outside the deployment domain)
              − land_penalty   · (# WECs on cells flagged as non-ocean)

    ``valid_mask_fn``   : optional ``callable(pos_xy) -> bool`` returning
                          True if the [x, y] (metres) position is over
                          water (SWAN cell with finite, non-land data).
                          If None, only the spacing + bounds penalties apply.
    """
    if seed is not None:
        np.random.seed(seed)

    nv = 2 * n_wecs
    xm, ym = domain
    lo = np.tile([0.0, 0.0], n_wecs)
    hi = np.tile([float(xm), float(ym)], n_wecs)

    pop = np.column_stack(
        [np.random.uniform(lo[j], hi[j], pop_size) for j in range(nv)])
    hist = np.zeros(n_gen)

    def fit(ch):
        pos = ch.reshape(n_wecs, 2)
        # --- out-of-bounds count (clamp for power calc, but still penalise) ---
        oob = int(((pos[:, 0] < 0) | (pos[:, 0] > xm)
                 | (pos[:, 1] < 0) | (pos[:, 1] > ym)).sum())
        # --- on-land count (after clamping so the query is valid) ---
        pos_clamped = np.clip(pos, [0.0, 0.0], [xm, ym])
        if valid_mask_fn is not None:
            on_land = sum(1 for p in pos_clamped if not bool(valid_mask_fn(p)))
        else:
            on_land = 0
        pw, _ = array_power(pos_clamped, k, P_field)
        return (pw
                - 1e4 * spacing_pen(pos_clamped, d_min)
                - bounds_penalty * oob
                - land_penalty   * on_land)

    for gen in range(n_gen):
        sc = np.array([fit(ind) for ind in pop])
        hist[gen] = sc.max()
        np_ = max(pop_size // 5, 2)
        parents = pop[np.argsort(sc)[::-1][:np_]].copy()
        no = pop_size - np_
        off = np.empty((no, nv))
        for i in range(no):
            p1, p2 = parents[i % np_], parents[(i + 1) % np_]
            off[i] = np.where(np.random.rand(nv) < .5, p1, p2)
        for i in range(no):
            for j in range(nv):
                if np.random.rand() < mut_rate:
                    off[i, j] = np.random.uniform(lo[j], hi[j])
        pop[:np_] = parents
        pop[np_:] = off

    sc = np.array([fit(ind) for ind in pop])
    bi = np.argmax(sc)
    bp = pop[bi].reshape(n_wecs, 2)
    bp = np.clip(bp, [0.0, 0.0], [float(xm), float(ym)])
    _, qa = array_power(bp, k, P_field)
    return bp, sc[bi], qa, hist


## 4. Execute the Full SWAN-Coupled Pipeline

This cell runs the complete workflow:
1. Check SWAN availability
2. Inject BLOCK output commands into the SWAN input
3. Run SWAN to compute the 2-D wave field
4. Read the `.MAT` block output and build a wave power interpolator
5. Run the GA using the interpolated field

**Note:** This requires SWAN to be installed. Without SWAN, the cell will print an error.

**Geometry.** The SWAN grid may be rotated (California Bay: 22.5 deg), so the wave-power field is interpolated in grid-index space and positions are handled in a local east/north metric frame. The GA searches a **1000 m x 500 m deployment window** (DEPLOY_W x DEPLOY_H) centred on the AHP-TOPSIS best site; if that window is mostly land or outside the grid, it is re-centred on the highest-power ocean cell. WEC positions are reported both in window metres and in lon/lat.


In [ ]:
print("=" * 65)
print(f"Live-Coupled SWAN + GA Array Optimisation  --  {cfg.display_name}")
print("=" * 65)

# Array deployment window (metres, east x north) centred on the best
# AHP-TOPSIS site. Same size as the standalone GA notebook.
DEPLOY_W, DEPLOY_H = 1000.0, 500.0

# --- 0. Check SWAN ---
if not swan_runner.check_swan_available(SWAN_CMD):
    print(f"\n  WARNING: '{SWAN_CMD}' not found.")
    print("  Install SWAN and re-run. See the repository README.")
    print("\n  To run the standalone GA without SWAN, use GA_array.ipynb")
else:
    # --- 1. Prepare SWAN with BLOCK output ---
    print("\n[1/4] Preparing SWAN input with block output...")
    # Output window: start 3 h after the region's simulation start, every 6 h
    block_start = (cfg.t_start + __import__('datetime').timedelta(hours=3)
                   ).strftime('%Y%m%d.%H%M%S')
    extra_cmds = [
        f"BLOCK 'COMPGRID' NOHEAD '{BLOCK_FILE}' LAY-OUT 3 "
        f"XP YP HSIGN RTP DIR DEPTH OUTPUT {block_start} 6 HR",
    ]
    swan_runner.prepare_input(
        INPUT_TEMPLATE, os.path.join(MODEL_DIR, 'ga_run.swn'), extra_cmds)

    # --- 2. Run SWAN ---
    print("[2/4] Running SWAN (may take several minutes)...")
    swan_runner.run_swan(MODEL_DIR, input_name='ga_run',
                         swan_cmd=SWAN_CMD, nproc=SWAN_NPROC)

    # --- 3. Read SWAN block output ---
    print("[3/4] Reading SWAN block output...")
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mat = swan_runner.read_mat_block(os.path.join(MODEL_DIR, BLOCK_FILE))

    Xp = mat['Xp']
    Yp = mat['Yp']
    Hs_all = swan_runner.extract_block_variable(mat, 'Hsig')
    Tp_all = swan_runner.extract_block_variable(mat, 'RTpeak')

    # Mask SWAN sentinel values (-9 for Hs/Tp) → NaN before averaging
    Hs_all = np.where(Hs_all < 0, np.nan, Hs_all)
    Tp_all = np.where(Tp_all < 0, np.nan, Tp_all)

    Hs_mean = np.nanmean(Hs_all, axis=0)
    Tp_mean = np.nanmean(Tp_all, axis=0)

    # A cell is "ocean" if either mean is finite AND positive (SWAN masks land as -9/NaN)
    ocean_mask_2d = np.isfinite(Hs_mean) & np.isfinite(Tp_mean) & (Hs_mean > 0)

    # Wave power field (kW/m)
    rho_g2 = 1025 * 9.81 ** 2 / (64 * np.pi)
    P_map = rho_g2 * np.nan_to_num(Hs_mean, nan=0.0) ** 2 \
          * np.nan_to_num(Tp_mean, nan=0.0) / 1000

    capture_width = 5.0  # metres
    P_wec_map = np.where(ocean_mask_2d, P_map * capture_width, 0.0)
    Tp_fill = np.where(ocean_mask_2d, np.nan_to_num(Tp_mean, nan=0.0), 0.0)

    # --- Geometry --------------------------------------------------------
    # The CGRID can be rotated (California: 22.5 deg), so lon AND lat change
    # along each array axis -- Xp[0, :] / Yp[:, 0] are not valid axes. Grid
    # nodes are regular in *index* space, so we interpolate there and do all
    # distance maths in a local east/north metric frame (equirectangular
    # about the CGRID origin; fine for a ~1 km deployment window).
    ny, nx = Xp.shape
    lon0, lat0 = float(Xp[0, 0]), float(Yp[0, 0])      # CGRID origin
    kx = 111320.0 * np.cos(np.radians(float(np.mean(Yp))))
    ky = 110540.0

    def lonlat_to_en(lon, lat):
        return np.array([(lon - lon0) * kx, (lat - lat0) * ky], dtype=float)

    def en_to_lonlat(e, n):
        return float(lon0 + e / kx), float(lat0 + n / ky)

    # Linear map from grid indices to metres: (east, north) = A @ (ix, iy)
    A = np.column_stack([lonlat_to_en(Xp[0, -1], Yp[0, -1]) / (nx - 1),
                         lonlat_to_en(Xp[-1, 0], Yp[-1, 0]) / (ny - 1)])
    A_inv = np.linalg.inv(A)
    _axes = (np.arange(ny, dtype=float), np.arange(nx, dtype=float))
    _kw = dict(bounds_error=False, fill_value=0.0)
    P_interp  = RegularGridInterpolator(_axes, P_wec_map, method='linear', **_kw)
    oc_interp = RegularGridInterpolator(_axes, ocean_mask_2d.astype(float),
                                        method='nearest', **_kw)
    Tp_interp = RegularGridInterpolator(_axes, Tp_fill, method='linear', **_kw)

    def _idx(en):
        ix, iy = A_inv @ np.asarray(en, dtype=float)
        return [[iy, ix]]

    # --- Deployment window around the AHP-TOPSIS best site ----------------
    centre = lonlat_to_en(site['lon'], site['lat'])

    def _ocean_fraction(c):
        es = c[0] + np.linspace(-DEPLOY_W / 2, DEPLOY_W / 2, 11)
        ns = c[1] + np.linspace(-DEPLOY_H / 2, DEPLOY_H / 2, 6)
        return float(np.mean([oc_interp(_idx((e, n)))[0] > 0.5
                              for e in es for n in ns]))

    if _ocean_fraction(centre) < 0.5:
        j_max, i_max = np.unravel_index(np.argmax(P_wec_map), P_wec_map.shape)
        centre = A @ np.array([i_max, j_max], dtype=float)
        print("       Best-site window is mostly land / outside the grid -> "
              "re-centred on the highest-power ocean cell.")
    origin = centre - np.array([DEPLOY_W / 2, DEPLOY_H / 2])
    c_lon, c_lat = en_to_lonlat(*centre)

    def P_field(pos_xy):
        """[x, y] metres inside the window -> isolated WEC power (kW)."""
        return float(P_interp(_idx(origin + np.asarray(pos_xy, float)))[0])

    def is_ocean(pos_xy):
        """Nearest-neighbour ocean/land flag at [x, y] metres in the window."""
        return float(oc_interp(_idx(origin + np.asarray(pos_xy, float)))[0]) > 0.5

    def window_to_lonlat(pos_xy):
        return en_to_lonlat(*(origin + np.asarray(pos_xy, float)))

    # Local dominant period at the window centre (fallback: ocean mean)
    Tp_dom = float(Tp_interp(_idx(centre))[0])
    if not np.isfinite(Tp_dom) or Tp_dom <= 0:
        Tp_dom = (float(np.nanmean(Tp_mean[ocean_mask_2d]))
                  if ocean_mask_2d.any() else 8.0)
    wavelength = 1.56 * Tp_dom ** 2
    k = 2 * np.pi / wavelength
    domain_m = (DEPLOY_W, DEPLOY_H)

    # Summary
    print(f"       Grid: {nx}x{ny} nodes (rotation handled in index space)")
    print(f"       Window: {DEPLOY_W:.0f} x {DEPLOY_H:.0f} m (east x north) "
          f"centred at ({c_lat:.3f}N, {c_lon:.3f}E)")
    print(f"       Tp_dom={Tp_dom:.1f} s  lambda={wavelength:.0f} m  k={k:.4f}")
    print(f"       Ocean cells: {int(ocean_mask_2d.sum())} / {ocean_mask_2d.size}")
    print(f"       P_wec at window centre = {P_field(np.array(domain_m) / 2):.1f} kW, "
          f"max in grid = {P_wec_map.max():.1f} kW")

    # --- 4. GA ---
    print("[4/4] Running GA (with on-land / out-of-bounds penalties)...")
    n_wecs = 5
    d_min  = 100

    pos, best_fit, q_arr, hist = run_ga(
        n_wecs, P_field, k, domain_m, d_min,
        valid_mask_fn=is_ocean, seed=42,
    )

    total_p, _ = array_power(pos, k, P_field)
    dists = [np.linalg.norm(pos[i] - pos[j])
             for i in range(n_wecs) for j in range(i + 1, n_wecs)]
    on_land = sum(1 for p in pos if not is_ocean(p))
    wec_lonlat = [window_to_lonlat(p) for p in pos]

    print(f"\n  {'='*50}")
    print(f"  RESULTS (SWAN-resolved wave field, {cfg.display_name})")
    print(f"  {'='*50}")
    print(f"    q-factor     : {q_arr:.4f}")
    print(f"    Total power  : {total_p:.1f} kW")
    print(f"    Min spacing  : {min(dists):.1f} m")
    print(f"    On-land WECs : {on_land}  (should be 0)")
    for i, ((x, y), (lo, la)) in enumerate(zip(pos, wec_lonlat)):
        print(f"    WEC {i+1}: ({x:7.1f}, {y:7.1f}) m  "
              f"({la:.5f}N, {lo:.5f}E)  [{'ocean' if is_ocean([x, y]) else 'LAND'}]")
    # --- 5. Persist best layout to figs/04_optimization/<region>/ ---
    out = {
        'region':   cfg.key,
        'n_wecs':   int(n_wecs),
        'q_factor': float(q_arr),
        'total_kw': float(total_p),
        'min_spacing_m': float(min(dists)),
        'on_land':  int(on_land),
        'window_m': [float(DEPLOY_W), float(DEPLOY_H)],
        'window_centre': {'lon': c_lon, 'lat': c_lat},
        'positions_m': [[float(x), float(y)] for x, y in pos],
        'positions_lonlat': [[lo, la] for lo, la in wec_lonlat],
    }
    out_path = os.path.join(FIGS_DIR, f'best_array_{cfg.key}.json')
    with open(out_path, 'w') as _f:
        json.dump(out, _f, indent=2)
    print(f'    Saved best layout -> {out_path}')


## 5. Visualization

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # Isolated WEC power inside the deployment window as a backdrop
    gx = np.linspace(0, domain_m[0], 101)
    gy = np.linspace(0, domain_m[1], 51)
    PW = np.array([[P_field((x, y)) for x in gx] for y in gy])
    im = ax1.imshow(PW, origin='lower', extent=(0, domain_m[0], 0, domain_m[1]),
                    cmap='viridis', alpha=0.6)
    fig.colorbar(im, ax=ax1, shrink=0.8, label='Isolated WEC power (kW)')
    ax1.scatter(pos[:, 0], pos[:, 1], s=250, c='dodgerblue',
                edgecolors='navy', linewidths=1.5, zorder=5, label='WEC')
    for i, (x, y) in enumerate(pos):
        ax1.annotate(f' {i+1}', (x, y), fontsize=10, fontweight='bold')
    ax1.add_patch(Rectangle((0, 0), domain_m[0], domain_m[1],
                             lw=1.5, ec='grey', fc='none', ls='--'))
    ax1.set_xlim(-50, domain_m[0] + 50)
    ax1.set_ylim(-50, domain_m[1] + 50)
    ax1.set_aspect('equal')
    ax1.set_xlabel('x east (m)'); ax1.set_ylabel('y north (m)')
    ax1.set_title(f"SWAN-Coupled Array Layout - {cfg.display_name} ({n_wecs} WECs)\n"
                  f"centre ({c_lat:.3f}N, {c_lon:.3f}E)  q={q_arr:.3f}  "
                  f"P={total_p:.0f} kW  spacing>={min(dists):.0f} m")
    ax1.legend(); ax1.grid(True, alpha=.3)

    # Convergence plot
    ax2.plot(hist, color='darkblue', lw=1.2)
    ax2.set_xlabel('Generation'); ax2.set_ylabel('Best Fitness (kW)')
    ax2.set_title(f'GA Convergence - {cfg.display_name}')
    ax2.grid(True, alpha=.3)

    plt.tight_layout()
    _fname = f"ga_array_{cfg.key}_cell12.png"
    plt.savefig(os.path.join(FIGS_DIR, _fname), dpi=140, bbox_inches='tight')
    plt.show()
except NameError:
    print("Results not available (SWAN was not run). Skipping plot.")


## Summary

This notebook demonstrated the complete SWAN-coupled GA array optimization:

1. **SWAN resolves** the 2-D nearshore wave field (Hs, Tp, direction, depth)
2. A **wave power interpolator** maps any (x, y) position to the local isolated WEC power
3. The **GA fitness function** combines spatially varying power with $q$-factor interactions and spacing constraints
4. The optimizer finds WEC positions that **exploit local wave energy hotspots** and **constructive interference**

Compared to the standalone GA (uniform field), the SWAN-coupled version produces layouts that are sensitive to bathymetric focusing, wave shadowing by headlands, and other nearshore effects that a uniform assumption would miss.